#  Week 2, Day 2 — May 26, 2026
## Dictionary-Based Text Mismatch Cleaning

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt, os, json, warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {
    "raw":            BASE_DIR + "/data/raw",
    "processed":      BASE_DIR + "/data/processed",
    "metadata":       BASE_DIR + "/data/metadata",
    "notebooks":      BASE_DIR + "/notebooks",
    "visualizations": BASE_DIR + "/visualizations"
}
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 5, 30, 65, 95
YEAR_MIN, YEAR_MAX = 2015, 2026

In [ ]:
df_plastic  = pd.read_csv(DIRS['processed'] + '/plastic_filtered.csv', parse_dates=['Date'])
df_species  = pd.read_csv(DIRS['processed'] + '/species_clean_v1.csv')
print(f'Loaded: plastic={df_plastic.shape}, species={df_species.shape}')

## Step 1: Normalize Plastic_Type — Shorten Verbose Names

In [ ]:
print('Before normalization:')
print(df_plastic['Plastic_Type'].value_counts())

plastic_type_map = {
    'Polyethylene Terephthalate (PET)': 'PET',
    'Polyethylene (PE)':                'PE',
    'Polystyrene (PS)':                 'PS',
    'Polypropylene (PP)':               'PP',
    'Polyvinyl Chloride (PVC)':         'PVC',
}

df_plastic_clean = df_plastic.copy()
df_plastic_clean['Plastic_Type'] = df_plastic_clean['Plastic_Type'].map(plastic_type_map)
unmapped = df_plastic_clean['Plastic_Type'].isnull().sum()

print(f'\nAfter normalization:')
print(df_plastic_clean['Plastic_Type'].value_counts())
print(f'Unmapped values: {unmapped}  (should be 0) ' if unmapped == 0 else f' {unmapped} unmapped!')

## Step 2: Normalize observation_type in Species Data

In [ ]:
print('Before normalization:')
print(df_species['observation_type'].value_counts())

obs_type_map = {
    'Scientific Record':  'Scientific',
    'PRESERVED_SPECIMEN': 'Scientific',
    'MATERIAL_CITATION':  'Scientific',
    'MATERIAL_SAMPLE':    'Scientific',
    'OCCURRENCE':         'Occurrence',
    'HUMAN_OBSERVATION':  'Human_Observation',
    'MARINE_OBS':         'Human_Observation',
    'UK_RECORD':          'Human_Observation',
    'OBSERVATION':        'Human_Observation',
    'MACHINE_OBSERVATION':'Machine_Observation',
    'REPTILE_CITIZEN':    'Citizen_Science',
    'CITIZEN_SCIENCE':    'Citizen_Science',
}

df_species_clean = df_species.copy()
df_species_clean['observation_type'] = df_species_clean['observation_type'].map(obs_type_map)

print('\nAfter normalization:')
print(df_species_clean['observation_type'].value_counts())

## Step 3: Handle NBN-Atlas-UK Records

In [ ]:
# 630 UK records in an India species dataset — check if within bbox
uk_records = df_species_clean[df_species_clean['data_source'] == 'NBN-Atlas-UK']
print(f'NBN-Atlas-UK records: {len(uk_records)}')
print(f'  Lat range: {uk_records["latitude"].min():.2f} to {uk_records["latitude"].max():.2f}')
print(f'  Lon range: {uk_records["longitude"].min():.2f} to {uk_records["longitude"].max():.2f}')
print()
# Check if any fall outside the already-applied bounding box
outside = uk_records[
    ~(uk_records['latitude'].between(LAT_MIN,LAT_MAX) &
      uk_records['longitude'].between(LON_MIN,LON_MAX))
]
print(f'UK records OUTSIDE bounding box: {len(outside)}')
print(f'UK records INSIDE bounding box: {len(uk_records)-len(outside)}')
print()
print('NOTE: Since we already applied bbox filter on Day 2, all remaining')
print('NBN-Atlas-UK records are within lat 5-30N, lon 65-95E.')
print('Keeping them — they likely represent marine observations near Indian territory.')

In [ ]:
# Save
df_plastic_clean.to_csv(DIRS['processed'] + '/plastic_clean_v1.csv', index=False)
df_species_clean.to_csv(DIRS['processed'] + '/species_clean_v2.csv', index=False)
print('Saved plastic_clean_v1.csv and species_clean_v2.csv ✅')
print(f'  plastic_clean_v1: Plastic_Type now uses abbreviations: {df_plastic_clean["Plastic_Type"].unique().tolist()}')
print(f'  species_clean_v2: observation_type now has {df_species_clean["observation_type"].nunique()} categories')

## ✅ Day 2 Week 2 Complete — May 26, 2026

**Next: May 27 — Unit Standardization**